# M05 · Statistical seismology: b-value, completeness, aftershock decay

**Part 1 · week 2 · about 10 h core + 2 h stretch**

## 한국어 요약

이 모듈에서는 통계지진학의 기본 도구를 직접 만들어 봅니다. 구텐베르크-리히터 법칙을 따르는 연습용 규모를 만들고, 규모 구간 폭(Δm)을 반영한 최대가능도 방법으로 b값과 그 불확실성(Shi-Bolt 식)을 계산하며, 최대 곡률법으로 완전성 규모(Mc)를 구합니다. 규모 차이로 b값을 구하는 b-positive 방법과, 여진 발생률이 시간에 따라 줄어드는 오모리-우츠 법칙의 최대가능도 맞춤도 구현합니다. 이렇게 만든 함수들을 2016년 경주 지진(ML 5.8)의 여진에 적용하여, 직접 정한 Mc에서 b값, b-positive, p값을 구하고 그림으로 나타낸 뒤, 발표된 연구 결과와 비교합니다. 심화 과제에서는 SeismoStats의 Gardner-Knopoff 디클러스터링으로 2018-2026년 전국 지진 목록의 b값이 어떻게 달라지는지 살펴봅니다.

## What you will be able to do

- Simulate rounded Gutenberg-Richter magnitudes and estimate b by maximum likelihood with the half-bin correction and Shi and Bolt's uncertainty, agreeing with SeismoStats to one part in a million
- Estimate the magnitude of completeness by maximum curvature, test it on a practice catalog whose detection fades gradually, and explain why one b-value must never span magnitudes that the catalog did not measure the same way
- Compute b-positive from magnitude differences and compare it with the classic estimate, right after a mainshock and in complete catalogs
- Fit the Omori-Utsu law by maximum likelihood and describe the trade-off between c and p and the effect of the start time
- Choose and defend Mc, then report b with its uncertainty, b-positive and Omori p for the 2016 Gyeongju aftershocks in a figure and compare them with published values; in the stretch task, show how Gardner-Knopoff declustering changes the national b-value

## Before you start

- Modules: M01, M02, M04
- Reading: Mignan & Woessner (2012), the sections on the maximum-curvature and b-value-stability methods; Utsu, Ogata & Matsu'ura (1995), the parts on the modified Omori formula and its parameters
- Needs: catalog
- Key terms (see the glossary): prediction (예측), random seed (난수 시드), earthquake catalog (지진 목록, 지진 카탈로그), seismicity (지진활동), frequency-magnitude distribution (규모-빈도 분포), Gutenberg-Richter law (구텐베르크-리히터 법칙), b-value (b값), a-value (a값), local magnitude (ML; 국지 규모), magnitude bin width (Δm; 규모 구간 폭), exponential distribution (지수 분포), cumulative distribution function (CDF; 누적 분포 함수), maximum likelihood estimation (최대가능도 추정, 최대우도 추정), likelihood (가능도, 우도), bias of an estimator (추정량의 편향), standard error (표준 오차), Monte Carlo simulation (몬테카를로 시뮬레이션), magnitude of completeness (Mc; 완전성 규모), maximum curvature (MAXC; 최대 곡률법), b-value stability method (b값 안정성 방법), mainshock (본진), foreshock (전진), aftershock (여진), earthquake sequence (지진 시퀀스), short-term aftershock incompleteness (단기 여진 불완전성), b-positive (b-positive (양의 규모 차이 b값)), Omori-Utsu law (오모리-우츠 법칙), Poisson process (푸아송 과정), negative log-likelihood (음의 로그 가능도), numerical optimization (수치 최적화), parameter trade-off (매개변수 상충 관계), inverse transform sampling (역변환 표본추출), histogram (히스토그램), declustering (디클러스터링, 종속 지진 제거), background seismicity (배경 지진활동), nearest-neighbor distance (최근접 이웃 거리), ETAS model (ETAS 모형), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M05.yaml](../../../.sgtl/criteria/M05.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M05.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
SEED = 2016                    # random seed of the practice catalogs in this notebook
DM = 0.1                       # magnitude bin width: KMA gives ML with one decimal
MC_CORRECTION = 0.2            # added to the maximum-curvature Mc (Woessner and Wiemer 2005)
N_REPEAT = 500                 # how many practice catalogs the Monte Carlo demos draw
SEQUENCE = "gyeongju2016"      # the named sequence of task 6 (st.data.sequence)
GJ_DMC = 0.2                   # task 6: smallest magnitude difference used by b-positive
GJ_T_START_DAYS = 0.1          # task 6: start of the Omori-Utsu fit, days after the mainshock
GJ_T_END_DAYS = 365.0          # task 6: end of the Omori-Utsu fit, days after the mainshock
NATIONAL_YEARS = (2018, 2026)  # task 7: first and last UTC year of the national catalog
NATIONAL_MC = 2.1              # task 7: completeness magnitude for the national b-value
FIGURE_DIR = "figures"         # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M05")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

Path(FIGURE_DIR).mkdir(exist_ok=True)
ML = st.plot.ML_LABEL                 # the magnitude label of the house style
rng = np.random.default_rng(SEED)     # one generator for all demos in this notebook

## How this notebook works

The same five steps as in M01 and M02, for every task: read the **Contract**, write a **prediction**
(glossary `prediction`), write your code under `# YOUR CODE HERE`, run the check, and answer the **Explain**
prompt in 3-5 sentences of your own. The checks of tasks 1-5 draw fresh practice catalogs from a
**random seed** (glossary `random_seed`) made from your user name; `st.reroll("M05-T1")` draws new ones.
Tasks 6 and 7 run your functions on the KMA catalog and never print the values they expect.

A Python **object** in this notebook that holds random numbers is a numpy **Generator**, made with
`np.random.default_rng(seed)`. Pass it around and draw from it (`rng.exponential(...)`, `rng.random(...)`),
so that the same seed always gives the same "random" numbers.

You may ask an AI tool to explain an error message, a formula or a concept. It may not write the task
functions, your predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI
conversations in `work/ai_log/M05.md`. After a kernel restart, rerun the parameters, setup and completed shared/core
cells in order so their variables and functions exist again. Full **Run All Cells** also executes optional tasks;
`bin/sgtl submit M05 --apply` validates core work, with optional work added by `--include-optional`.

This module uses SeismoStats, a Python package for statistical seismology from the Swiss Seismological
Service at ETH Zurich (Mirwald et al. 2026), as an independent check of your functions and, in task 7, as a
tool. Never draw maps with SeismoStats: in this course, maps are made with PyGMT (M02).

## Part 1 · Counting earthquakes by size (about 60 min, then task 1)

An **earthquake catalog** (glossary `earthquake_catalog`) lists the **seismicity** (glossary `seismicity`) of a
region: origin time, location and magnitude of every earthquake the network detected. Count the events in
each magnitude bin and you get the **frequency-magnitude distribution** (glossary
`frequency_magnitude_distribution`, FMD for short). Plotted on a logarithmic count axis, the FMD of most
regions falls along a straight line. That line is the **Gutenberg-Richter law** (glossary
`gutenberg_richter_law`; Gutenberg and Richter 1944):

$$\log_{10} N(\ge M) = a - b\,M,$$

where $N(\ge M)$ is the number of events with magnitude $M$ or larger. The **b-value** (glossary `b_value`)
is the slope: with $b = 1$, every step of one magnitude unit has ten times fewer events. The **a-value**
(glossary `a_value`) sets how many events there are in all. Most regions have $b$ between about 0.8 and 1.2.

Two facts make the law usable:

1. Above a threshold magnitude $M_c$, the magnitudes behave like draws from an **exponential distribution**
   (glossary `exponential_distribution`): $P(M \ge m) = 10^{-b\,(m - M_c)} = e^{-\beta\,(m - M_c)}$, with
   $\beta = b \ln 10$. So $b$ (base 10) and $\beta$ (base $e$) carry the same information; they differ by the
   factor $\ln 10 \approx 2.303$.
2. Catalogs **round** magnitudes. KMA gives **local magnitude** ML (glossary `local_magnitude`) with one
   decimal, so the **magnitude bin width** (glossary `magnitude_binning`) is $\Delta m$ = 0.1: a listed 2.3
   stands for any magnitude from 2.25 to 2.35. Every estimator in this module has to take this into account.

Look at the FMD of the KMA catalog since 2018. Where does it follow a straight line, and where not?

In [ ]:
catalog = st.data.kma_catalog()
since_2018 = catalog[catalog["origin_utc"] >= pd.Timestamp("2018-01-01", tz="UTC")]
per_bin = since_2018["ml"].round(1).value_counts().sort_index()          # events in each 0.1 bin
n_at_or_above = per_bin.iloc[::-1].cumsum().iloc[::-1]                     # events at or above each bin
print(f"{len(since_2018)} events since 2018 | source: {catalog.attrs['source']}")
print(per_bin.loc[1.5:2.3].to_string())

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.semilogy(per_bin.index, per_bin.to_numpy(), "o", ms=4, color="tab:blue", label="Per bin")
ax.semilogy(n_at_or_above.index, n_at_or_above.to_numpy(), "s", ms=4, mfc="none", color="0.25",
            label="At or above")
ax.set(xlabel=f"Local magnitude, {ML}", ylabel="Number of events", title="KMA catalog since 2018 (UTC)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

Above about ML 2, the per-bin counts fall roughly along a straight line: the Gutenberg-Richter law. Below
about ML 1, they fall again, because the network does not detect every small earthquake. And just below
ML 2.0 something odd happens: the bins 1.8 and 1.9 hold several times more events than 2.0 and 2.1 — the
pile-up you measured in M04. Its cause is not known. We come back to it in Part 3.

To test estimators we need practice catalogs whose true $b$ we know. Task 1 makes them.

### M05-T1 · Practice magnitudes from the Gutenberg-Richter law · core · about 60 min

**Goal:** write `simulate_gr(n, b, mc, dm, rng)`: `n` magnitudes that follow the Gutenberg-Richter law with
slope `b` above `mc`, rounded the way a catalog rounds them.

**Given:** above a threshold $m_0$, magnitudes follow the exponential distribution
$P(M \ge m) = e^{-\beta (m - m_0)}$ with $\beta = b \ln 10$. numpy draws such numbers:
`rng.exponential(scale, size)` returns exponential draws with mean `scale`, and the mean of this
distribution is $1/\beta$.

**Contract:**
- Inputs: `n` (int), `b` (float, base-10 slope), `mc` (float, the smallest rounded magnitude, a whole number of
  bins: `mc / dm` is an integer), `dm` (float, bin width, for example 0.1), `rng` (a numpy Generator).
- Output: a numpy array of `n` floats, each one of `mc`, `mc + dm`, `mc + 2*dm`, ... (up to tiny float
  rounding).
- Rounding as in a catalog: the bin at a value $m_k$ holds every true magnitude from $m_k - dm/2$ to
  $m_k + dm/2$, and that includes the bin at `mc`: it must receive its full share of events.
- Draw every random number from `rng` itself, so that the same seed gives the same magnitudes (no generator of
  your own and no `np.random` function inside the function: the check refuses them).
- Make continuous (unrounded) magnitudes first and round them yourself, as a catalog does. Drawing whole bin
  numbers directly (`rng.geometric`, `rng.integers`, `rng.choice` and the like) skips the point of this task,
  and the check refuses it.
- Do not use SeismoStats or `sgtl_tutorial` inside the function (the check refuses them).

**Done when** `st.check("M05-T1", func=simulate_gr)` passes: 6 fresh settings (b from 0.6 to 1.5; dm 0.1, 0.2,
0.05 or 0.01), 200,000 magnitudes each, are compared with the share of events that the Gutenberg-Richter law
puts into each bin when magnitudes are rounded correctly (two statistical tests on the counts per bin;
thresholds in criteria/M05.yaml). Then the same seed must give the same values.

**Stuck?** `st.hint("M05-T1", 1)`

In [ ]:
# Before you code: where must the continuous draws start so that the bin at mc gets its full share?
st.predict("M05-T1", """

""")

In [ ]:
def simulate_gr(n, b, mc, dm, rng):
    """Gutenberg-Richter magnitudes rounded to steps of dm.

    n   : int, how many magnitudes
    b   : float, b-value (base-10 slope)
    mc  : float, smallest rounded magnitude (a whole number of bins)
    dm  : float, bin width
    rng : numpy Generator; every random draw comes from it
    returns : numpy array of n floats, each mc + k*dm with k = 0, 1, 2, ...
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T1: write your code here")

In [ ]:
st.check("M05-T1", func=simulate_gr)

**Explain (3-5 sentences):** Where do your continuous draws start, and why there? What would the FMD below
look like if they started somewhere else? How did you turn `b` into the numpy argument, and what goes wrong
if you use `b` itself? Why must the function draw from `rng` and not from a generator of its own?

Use it: 100,000 practice magnitudes with $b = 1$ above $M_c = 1.0$. The "at or above" counts should fall by
a factor of ten per magnitude unit.

In [ ]:
practice = simulate_gr(100_000, 1.0, 1.0, DM, rng)
values, counts = np.unique(practice, return_counts=True)
above_counts = counts[::-1].cumsum()[::-1]
print("Ratio of events at or above M and at or above M + 1:",
      np.round(above_counts[:5] / above_counts[10:15], 2))

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.semilogy(values, counts, "o", ms=4, color="tab:blue", label="Per bin")
ax.semilogy(values, above_counts, "s", ms=4, mfc="none", color="0.25", label="At or above")
ax.set(xlabel="Magnitude", ylabel="Number of events", title="Practice catalog, b = 1")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

## Part 2 · Estimating b by maximum likelihood (about 60 min, then task 2)

Do not fit a straight line through the FMD with least squares: the counts of the largest bins are tiny and
noisy, the points of a cumulative curve are not independent, and the answer depends on how you bin. The
standard method is **maximum likelihood** (glossary `maximum_likelihood`). The **likelihood** (glossary
`likelihood`) is the probability of the observed magnitudes for a trial value of $b$; the maximum-likelihood
estimate is the $b$ that makes the data most probable. For continuous magnitudes above $M_c$ the answer is
short (Aki 1965; Utsu 1965):

$$\hat b = \frac{\log_{10} e}{\bar M - M_c},$$

where $\bar M$ is the mean magnitude of the events at or above $M_c$. A larger mean means relatively more large
events, so a smaller $b$.

**Rounded magnitudes need a correction.** The bin at $M_c$ holds magnitudes down to $M_c - \Delta m/2$, so the
rounded catalog starts half a bin lower than its smallest listed value. Utsu (1965) replaced $M_c$ by
$M_c - \Delta m/2$:

$$\hat b = \frac{\log_{10} e}{\bar M - (M_c - \Delta m/2)}.$$

Without it, the estimate is biased: on average it comes out wrong, here too high. A **bias** (glossary
`estimator_bias`) is such a systematic error. The exact maximum-likelihood formula for rounded magnitudes
was derived by Tinti and Mulargia (1987):

$$\hat b = \frac{1}{\Delta m \ln 10}\,\ln\!\left(1 + \frac{\Delta m}{\bar M - M_c}\right).$$

SeismoStats calls it `ClassicBValueEstimator`; its `UtsuBValueEstimator` is the corrected Aki-Utsu formula.
At $\Delta m$ = 0.1 the two differ by about 0.4 % (Marzocchi and Sandri 2003 review these formulas).

**How certain is b?** The **standard error** (glossary `standard_error`) of an estimate says how much it
would scatter if we could repeat the experiment. Shi and Bolt (1982) give

$$\sigma_b = \ln 10 \cdot b^2 \sqrt{\frac{\sum_i (M_i - \bar M)^2}{n\,(n-1)}}$$

(they wrote 2.30 for $\ln 10$). For a large sample this is close to $b/\sqrt{n}$: with 100 events,
$\sigma_b \approx 0.1$. Small samples give uncertain b-values, and differences of 0.1 between two b-values
from 100 events each mean nothing.

### M05-T2 · b-value and its uncertainty · core · about 90 min

**Goal:** write `b_utsu(mags, mc, dm)`, the corrected Aki-Utsu estimate of the b-value, and
`shi_bolt(mags, b)`, its uncertainty.

**Given:** the second formula of Part 2 (with $M_c - \Delta m/2$) and the Shi-Bolt formula.

**Contract:**
- `b_utsu`: `mags` is a numpy array of rounded magnitudes (a whole number of bins), in any order; it may
  contain events below `mc`. Use only the events at or above `mc`. Rounded values are stored as floats, so
  2.0 may arrive as 1.9999999999999998: treat an event as "at or above `mc`" when `m >= mc - dm/2`.
  Return $b$ (base 10) as one float.
- `shi_bolt`: `mags` holds only the events used for $b$ (all at or above `mc`), `b` is their b-value. Return
  $\sigma_b$ as one float, with $\ln 10$ (the check also accepts the 2.30 of the paper).
- Do not use SeismoStats or `sgtl_tutorial` inside the functions.

**Done when** `st.check("M05-T2", func=b_utsu, shi_bolt=shi_bolt)` passes: 40 fresh practice catalogs
(30 to 2,500 events at or above `mc`, most with events below it, half with tiny float noise) agree with
SeismoStats' `UtsuBValueEstimator` to 1 part in a million, then 20 Shi-Bolt cases agree with SeismoStats'
`shi_bolt_confidence` to 0.2 %.

**Stuck?** `st.hint("M05-T2", 1)`

In [ ]:
# Before you code: at b = 1 and dm = 0.1, how large (in percent) will the error be without the half-bin
# correction, and is it larger or smaller than the scatter of b for 100 events and for 1,000 events?
st.predict("M05-T2", """

""")

In [ ]:
def b_utsu(mags, mc, dm):
    """Corrected Aki-Utsu maximum-likelihood b-value of rounded magnitudes.

    mags : numpy array of magnitudes (a whole number of bins; may include events below mc)
    mc   : float, completeness magnitude (a whole number of bins)
    dm   : float, bin width
    returns : float, b-value (base 10) of the events at or above mc
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T2: write your code here")


def shi_bolt(mags, b):
    """Shi and Bolt (1982) uncertainty of a b-value.

    mags : numpy array of the magnitudes used for b (all at or above mc)
    b    : float, their b-value
    returns : float, one standard error of b
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T2: write your code here")

In [ ]:
st.check("M05-T2", func=b_utsu, shi_bolt=shi_bolt)

**Explain (3-5 sentences):** Run the Monte Carlo cell below first. How does the scatter of $\hat b$ change
from 50 to 1,000 events, and how well does Shi and Bolt's $\sigma_b$ describe it? How large is the bias
without the half-bin correction, and how large is the difference to SeismoStats' exact binned
estimator? What would you say to a colleague who reports $b = 0.85$ in one area and $b = 1.0$ in another,
from 80 events each?

A **Monte Carlo simulation** (glossary `monte_carlo_simulation`) repeats a calculation on many random data
sets to see how the results scatter. Here: `N_REPEAT` practice catalogs with $b = 1$ for each size.

In [ ]:
true_b = 1.0
rows = []
for n_events in (50, 200, 1000):
    est, sig, plain = [], [], []
    for _ in range(N_REPEAT):
        mags = simulate_gr(n_events, true_b, 1.0, DM, rng)
        b_hat = b_utsu(mags, 1.0, DM)
        est.append(b_hat)
        sig.append(shi_bolt(mags, b_hat))
        plain.append(np.log10(np.e) / (mags.mean() - 1.0))     # no half-bin correction, on purpose
    rows.append({"events": n_events, "mean b": np.mean(est), "scatter of b (std)": np.std(est, ddof=1),
                 "mean Shi-Bolt sigma": np.mean(sig), "b over sqrt(n)": true_b / np.sqrt(n_events),
                 "mean b without correction": np.mean(plain)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# The exact binned estimator in SeismoStats, on one big practice catalog:
from seismostats.analysis import ClassicBValueEstimator, UtsuBValueEstimator

big = simulate_gr(200_000, true_b, 1.0, DM, rng)
b_mine = b_utsu(big, 1.0, DM)
b_exact = ClassicBValueEstimator().calculate(big, mc=1.0, delta_m=DM)
print(f"yours {b_mine:.4f} | SeismoStats Utsu {UtsuBValueEstimator().calculate(big, mc=1.0, delta_m=DM):.4f} | "
      f"SeismoStats Classic (Tinti-Mulargia) {b_exact:.4f} | difference {100 * (b_mine / b_exact - 1):+.2f} %")

## Part 3 · The magnitude of completeness (about 60 min, then task 3)

The estimators of Part 2 assume that no event at or above $M_c$ is missing from the catalog. The **magnitude of
completeness** $M_c$ (glossary `magnitude_of_completeness`) is the smallest magnitude for which that is
(almost) true. Below it, the network misses events, so the FMD bends down. If $M_c$ is set too low, the
estimate of $b$ is biased low; if it is set too high, we throw away data and $\sigma_b$ grows.

The simplest estimate is the **maximum curvature method** (glossary `maximum_curvature`; Wiemer and
Wyss 2000): take the bin with the most events, the peak of the per-bin FMD, which is a **histogram** (glossary
`histogram`). Because detection usually fades gradually below $M_c$, the peak sits a little below the true
$M_c$; Woessner and Wiemer (2005) add 0.2 (`MC_CORRECTION`). Mignan and Woessner (2012) review this and other
methods.

### M05-T3 · Completeness from the peak of the histogram · core · about 60 min

**Goal:** write `mc_maxc(mags, dm, correction=0.2)`: the maximum-curvature magnitude of completeness.

**Given:** the method above; `np.unique(..., return_counts=True)` counts how often each value occurs and
returns the values sorted.

**Contract:**
- `mags`: numpy array of rounded magnitudes (a whole number of bins, possibly with tiny float noise), any order.
- Count the events in bins of width `dm` centered on whole multiples of `dm` (the listed values themselves).
- Return the center of the bin with the most events plus `correction`, as one float. If several bins share the
  largest count, take the smallest magnitude.
- Do not use SeismoStats or `sgtl_tutorial` inside the function.

**Done when** `st.check("M05-T3", func=mc_maxc)` passes: 30 fresh practice catalogs with gradual detection
(and a few with a tie) agree with SeismoStats' `estimate_mc_maxc` to 1e-6.

**Stuck?** `st.hint("M05-T3", 1)`

In [ ]:
# Before you code: for a catalog whose detection fades slowly over a whole magnitude unit, will MAXC + 0.2 be
# above or below the magnitude where detection is really complete?
st.predict("M05-T3", """

""")

In [ ]:
def mc_maxc(mags, dm, correction=0.2):
    """Maximum-curvature magnitude of completeness (Wiemer and Wyss 2000) plus a correction.

    mags       : numpy array of rounded magnitudes
    dm         : float, bin width
    correction : float, added to the peak bin (0.2 after Woessner and Wiemer 2005)
    returns    : float, magnitude of completeness
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T3: write your code here")

In [ ]:
st.check("M05-T3", func=mc_maxc)

**Explain (3-5 sentences):** Use the demo below. Where is MAXC + 0.2 compared with the magnitude above which
99 % of the events are detected, and what does that do to $b$? What does the **b-value stability method**
(glossary `b_value_stability`) choose, and why can it do better here? Why should the bin centers be the listed
values and not the edges of an `np.histogram` call?

**Gradual detection.** A practice catalog with $b = 1$ from magnitude 0, of which an event of magnitude $m$ is
detected with probability `norm.cdf((m - 1.6) / 0.5)` (a smooth step: 50 % at 1.6, 99 % at about 2.8).
SeismoStats' `estimate_mc_b_stability` (the b-stability method; Woessner and Wiemer 2005) is shown for
comparison. Note: `estimate_mc_maxc` takes `fmd_bin=` (its docstring example passes `delta_m=`, which fails).

In [ ]:
import contextlib
import io
import warnings

from seismostats.analysis import estimate_mc_b_stability, estimate_mc_maxc

DETECT_50, DETECT_WIDTH = 1.6, 0.5        # detection: 50 % at 1.6, a smooth step of this width
full = simulate_gr(150_000, 1.0, 0.0, DM, rng)
seen = full[rng.random(full.size) < norm.cdf((full - DETECT_50) / DETECT_WIDTH)]
mc_peak = mc_maxc(seen, DM, MC_CORRECTION)
with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):   # keep the library quiet
    warnings.simplefilter("ignore")
    mc_stable, _ = estimate_mc_b_stability(seen, delta_m=DM)
print(f"{seen.size} detected events | yours {mc_peak:.1f} | SeismoStats MAXC "
      f"{estimate_mc_maxc(seen, fmd_bin=DM, correction_factor=MC_CORRECTION)[0]:.1f} | b-stability {mc_stable:.1f}")
print(f"b at MAXC + 0.2: {b_utsu(seen, mc_peak, DM):.2f} | b at the b-stability Mc: {b_utsu(seen, mc_stable, DM):.2f}"
      f" | 99 % detection at about {DETECT_50 + DETECT_WIDTH * norm.ppf(0.99):.1f}")

trial_mc = np.round(np.arange(1.0, 3.21, DM), 1)
trial_b = [b_utsu(seen, m0, DM) for m0 in trial_mc]
trial_s = [shi_bolt(seen[seen >= m0 - DM / 2], bb) for m0, bb in zip(trial_mc, trial_b)]
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.errorbar(trial_mc, trial_b, yerr=trial_s, fmt="o", ms=4, color="tab:blue", label="b at each trial Mc")
ax.axhline(1.0, color="0.4", ls="--", lw=1, label="True b")
ax.axvline(mc_peak, color="tab:red", ls=":", lw=1.2, label="MAXC + 0.2")
ax.axvline(mc_stable, color="tab:green", ls="-.", lw=1.2, label="b-stability")
ax.set(xlabel="Trial completeness magnitude", ylabel="b-value", title="Gradual detection")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), frameon=False, ncol=2, fontsize=9)
plt.show()

**Two traps in the KMA catalog.**

1. *Network epochs.* The network grew: before 2008 almost only events of ML 2.0 or more are listed; from 2008
   many smaller ones are (catalog notes). One $M_c$ for 1978-2026 is meaningless; estimate $M_c$ per epoch and
   compare rates only within an epoch or above a magnitude complete in all of them.
2. *The pile-up below ML 2.0.* Since 2018 the bins 1.8 and 1.9 hold several times the events of 2.0 and 2.1. Any
   $M_c$ below 2.0 makes the estimator average over this step. For a national b-value from the KMA catalog,
   use $M_c \ge 2.1$ and never fit one b-value across ML 2.0.

In [ ]:
epochs = {"1978-2007": ("1978-01-01", "2008-01-01"), "2008-2015": ("2008-01-01", "2016-01-01"),
          "2016-2026": ("2016-01-01", "2027-01-01")}
for name, (t0, t1) in epochs.items():
    part = catalog[(catalog["origin_utc"] >= pd.Timestamp(t0, tz="UTC"))
                   & (catalog["origin_utc"] < pd.Timestamp(t1, tz="UTC"))]["ml"].to_numpy()
    print(f"{name}: {part.size:5d} events, MAXC + {MC_CORRECTION} = {mc_maxc(part, DM, MC_CORRECTION):.1f}")

recent = since_2018["ml"].to_numpy()
for m0 in (1.5, 1.8, 2.1, 2.3):
    b0 = b_utsu(recent, m0, DM)
    used = recent[recent >= m0 - DM / 2]
    print(f"since 2018, Mc {m0:.1f}: {used.size:5d} events, b = {b0:.2f} +- {shi_bolt(used, b0):.2f}")

**Explorer E9 (optional, not checked).** A **widget** (glossary `widget`) is a control in a notebook, such as a
slider or a button, that runs code again when you move it. The explorer below redraws the frequency-magnitude
distribution of the KMA catalog since 2018 for the $M_c$ and the bin width you set with two sliders. It computes
nothing by itself: the counts come from your `fmd` of M04, $b$ and $\sigma_b$ from your `b_utsu` and `shi_bolt`,
and the dashed line from your `mc_maxc`. Copy your `fmd` from M04 into the cell first; without it you see the
magnitudes only. Things to try: move $M_c$ from 1.5 to 2.5, one bin at a time; set the bin width to 0.05 and to
0.2, which are not the step in which the catalog lists ML. What happens to the counts and to $b$, and why? The
explorer takes any magnitudes, for example `seen` from the gradual-detection demo, whose true $b$ is 1.

In [ ]:
# Optional explorer E9 (not checked). Paste your fmd function from M04 above this comment to see the counts.
your_fmd = globals().get("fmd")                # your fmd when this notebook has one, otherwise None
mags_to_explore = recent                       # or seen, the practice catalog of the gradual-detection demo
st.widgets.fmd_explorer(mags_to_explore, fmd=your_fmd, b_estimator=b_utsu, sigma_estimator=shi_bolt,
                        mc_estimator=mc_maxc, dm=DM)

## Part 4 · Aftershocks and b-positive (about 60 min, then task 4)

The largest event of an **earthquake sequence** (glossary `earthquake_sequence`) is the **mainshock**
(glossary `mainshock`); smaller events before it are **foreshocks** (glossary `foreshock`) and those after it
are **aftershocks** (glossary `aftershock`). Right after a large mainshock, its long, strong seismograms and the
flood of aftershocks hide the small events: for hours to days the catalog misses aftershocks it would list on a
quiet day. This **short-term aftershock incompleteness** (glossary `transient_incompleteness`) raises $M_c$
for a while. For southern California, Helmstetter et al. (2006) found roughly
$M_c(t) \approx M_{\text{main}} - 4.5 - 0.75\log_{10} t$, with $t$ in days.

**b-positive** (glossary `b_positive`; van der Elst 2021) was proposed for such catalogs. Take the magnitudes
in time order and their differences to the event before, $d_i = m_{i+1} - m_i$. Keep only the differences
$d \ge \delta_c$ (`dmc`; at least one bin). The kept differences follow an exponential distribution with the
same $b$ as the magnitudes, as long as an event that is larger than the one before it is recorded, which holds
even when the small events are being missed. The price: fewer usable numbers and a different distribution,
so b-positive scatters more than the classic estimate from the same complete catalog.

The differences of rounded magnitudes are rounded to steps of $\Delta m$ too, so we use the exact binned
formula of Tinti and Mulargia (1987), with $\delta_c$ in the place of $M_c$:

$$\hat b_+ = \frac{1}{\Delta m \ln 10}\,\ln\!\left(1 + \frac{\Delta m}{\bar d - \delta_c}\right),$$

where $\bar d$ is the mean of the kept differences.

### M05-T4 · b from magnitude differences · core · about 75 min

**Goal:** write `b_positive(mags, dm, dmc=None)`: the b-positive estimate of van der Elst (2021).

**Given:** the formula above.

**Contract:**
- `mags`: numpy array of rounded magnitudes **in time order** (earliest first), possibly with tiny float noise.
- Differences: each magnitude minus the one just before it in time. Keep the differences of at least `dmc`;
  `dmc` is a whole number of bins, and `None` means `dm`. Rounding noise can make a difference of exactly one
  bin look like 0.0999999999999996, so keep a difference when `d >= dmc - dm/2`.
- Return $\hat b_+$ from the kept differences with the formula above, as one float.
- Do not use SeismoStats or `sgtl_tutorial` inside the function.

**Done when** `st.check("M05-T4", func=b_positive)` passes: 30 fresh time-ordered practice catalogs with
short-term incompleteness (with `dmc` left out, or 1, 2 or 3 bins) agree with SeismoStats'
`BPositiveBValueEstimator` to 1 part in a million.

**Stuck?** `st.hint("M05-T4", 1)`

In [ ]:
# Before you code: in the first day after a mainshock, will the classic b at the long-term Mc be too high or
# too low, and will b-positive be closer to the truth?
st.predict("M05-T4", """

""")

In [ ]:
def b_positive(mags, dm, dmc=None):
    """b-positive (van der Elst 2021) from the positive magnitude differences of a time-ordered catalog.

    mags : numpy array of rounded magnitudes in time order (earliest first)
    dm   : float, bin width
    dmc  : float or None, smallest difference used (a whole number of bins; None means dm)
    returns : float, b-value (base 10)
    """
    # Step 1: decide the threshold, then take the differences of neighbors in time.
    # Step 2: keep the differences at or above the threshold (with the half-bin margin of the Contract).
    # Step 3: put the mean of the kept differences into the formula, with dmc in the place of Mc.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T4: write your code here")

In [ ]:
st.check("M05-T4", func=b_positive)

**Explain (3-5 sentences):** Use the demo below. In the first day, how far from the true $b = 1$ are the classic
estimate and b-positive, and why? How much more does b-positive scatter in complete catalogs of the same size?
Why must the magnitudes stay in time order, and what would sorting them do?

**Demo: a practice aftershock sequence with short-term incompleteness.** A mainshock of magnitude 6.0 at
$t = 0$; 20,000 aftershocks with $b = 1$ above magnitude 1.0, spread evenly in log time between 0.001 and 100
days (as the Omori-Utsu law of Part 5 does for $p = 1$); an aftershock is missed when its magnitude is below
Helmstetter's $M_c(t)$.

In [ ]:
n_after = 20_000
t_days = 10 ** rng.uniform(-3.0, 2.0, n_after)
order = np.argsort(t_days)
t_days = t_days[order]
m_true = simulate_gr(n_after, 1.0, 1.0, DM, rng)
mc_t = 6.0 - 4.5 - 0.75 * np.log10(t_days)
listed = m_true >= mc_t
print(f"{listed.sum()} of {n_after} aftershocks are listed")

for label, window in (("first day", t_days <= 1.0), ("days 10-100", t_days >= 10.0)):
    sub = m_true[listed & window]
    print(f"{label:12s}: {sub.size:5d} listed | classic b at Mc 1.0: {b_utsu(sub, 1.0, DM):.2f} | "
          f"b-positive (dmc 0.2): {b_positive(sub, DM, 0.2):.2f}")

# The price of b-positive: scatter in complete practice catalogs of 500 events.
classic, positive = [], []
for _ in range(N_REPEAT):
    mags = simulate_gr(500, 1.0, 1.0, DM, rng)
    classic.append(b_utsu(mags, 1.0, DM))
    positive.append(b_positive(mags, DM, 0.2))
print(f"500 complete events: std of classic b {np.std(classic, ddof=1):.3f}, "
      f"std of b-positive {np.std(positive, ddof=1):.3f}")

## Part 5 · Aftershock decay: the Omori-Utsu law (about 90 min, then task 5)

The rate of aftershocks falls with the time $t$ since the mainshock. The **Omori-Utsu law** (glossary
`omori_utsu_law`; Utsu et al. 1995 review its first century) says

$$\lambda(t) = \frac{K}{(t + c)^p},$$

in events per day for $t$ in days. $K$ sets the productivity, $p$ (usually 0.9-1.3) how fast the rate decays,
and $c$ (minutes to hours) keeps the rate finite at $t = 0$; part of $c$ often reflects the early
incompleteness of Part 4.

**The likelihood of event times.** Treat the aftershocks above $M_c$ as a **Poisson process** (glossary
`poisson_process`) whose rate changes with time. For events at times $t_1, \dots, t_n$ observed between
$t_s$ and $t_e$, the log-likelihood is (Ogata 1983)

$$\ln L = \sum_{i=1}^{n} \ln \lambda(t_i) - \int_{t_s}^{t_e} \lambda(t)\,dt .$$

The sum rewards a high rate where events happened; the integral, the number of events the model expects in the
window, penalizes a high rate everywhere. Optimizers minimize, so we work with the **negative
log-likelihood** (glossary `negative_log_likelihood`) $-\ln L$. The integral has a closed form:

$$\int_{t_s}^{t_e} \frac{K\,dt}{(t+c)^p} = \begin{cases} K\,\dfrac{(t_e+c)^{1-p} - (t_s+c)^{1-p}}{1-p}, & p \ne 1,\\[2ex]
  K \ln\dfrac{t_e + c}{t_s + c}, & p = 1. \end{cases}$$

Finding the minimum is **numerical optimization** (glossary `numerical_optimization`):
`scipy.optimize.minimize(f, x0, method="Nelder-Mead")` tries parameter values step by step from a starting
guess `x0`. Two tips: let the optimizer work on $\ln K$ and $\ln c$, so they stay positive, and start it from
several values of $c$ and keep the best result.

### M05-T5 · Fitting the aftershock decay · core · about 120 min

**Goal:** write `omori_negloglik(params, t, t_start, t_end)`, the negative log-likelihood of the Omori-Utsu law,
and `fit_omori(t, t_start, t_end)`, its maximum-likelihood fit.

**Given:** the log-likelihood and the integral above; `scipy.optimize.minimize` (imported in the setup cell).

**Contract:**
- `params`: `(K, c, p)`, a tuple or a numpy array: K in events per day, c in days, p without unit, all plain
  (not logarithms).
- `t`: numpy array of event times in days after the mainshock, all between `t_start` and `t_end` (days).
- `omori_negloglik` returns $-\ln L$ as one float. It must work for $p = 1$ exactly.
- `fit_omori` returns `(K, c, p)` (three floats, `K > 0`, `c > 0`) that minimize `omori_negloglik` for these
  times and this window.
- Do not use `sgtl_tutorial` inside the functions.

**Done when** `st.check("M05-T5", func=omori_negloglik, fit=fit_omori)` passes: 30 fresh parameter sets and
event lists (some with $p = 1$) agree with a numerical integration to 1 part in a million, then 4 practice
sequences drawn from known $(K, c, p)$ (800-3,000 events): your fit must explain the times at least as well
as the true parameters, and $p$ must be within 0.15 of the truth (a tolerance from a Monte Carlo study of the
fit).

**Stuck?** `st.hint("M05-T5", 1)`

In [ ]:
# Before you code: if you start the fit at t_start = 1 day instead of 0, which of K, c, p will become the most
# uncertain, and why?
st.predict("M05-T5", """

""")

In [ ]:
def omori_negloglik(params, t, t_start, t_end):
    """Negative log-likelihood of the Omori-Utsu rate K/(t + c)**p for event times t in [t_start, t_end].

    params : (K, c, p) with K in events per day, c in days, p without unit
    t      : numpy array of event times, days after the mainshock
    t_start, t_end : float, the observation window in days
    returns : float, minus the log-likelihood
    """
    # Step 1: the sum of the log-rates at the event times (log K minus p times log(t + c), for every event).
    # Step 2: the expected number of events in the window (the integral of the Given; p = 1 has its own form).
    # Step 3: minus the log-likelihood = expected number minus the sum of Step 1.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T5: write your code here")


def fit_omori(t, t_start, t_end):
    """Maximum-likelihood Omori-Utsu parameters (K, c, p) of event times t (days) in [t_start, t_end]."""
    # Step 1: write a small function of the free parameters (for example log c and p) that returns the
    #         negative log-likelihood; get K from the other two or let the optimizer handle log K as well.
    # Step 2: run scipy.optimize.minimize from several starting values of c and keep the lowest result.
    # Step 3: turn the winner back into plain K, c, p.
    # YOUR CODE HERE
    raise NotImplementedError("M05-T5: write your code here")

In [ ]:
st.check("M05-T5", func=omori_negloglik, fit=fit_omori)

**Explain (3-5 sentences):** Use the demo below. How do $c$ and $p$ change when the fit starts at 0.01, 0.1
and 1 day, and what does the contour plot say about why? Which number of the fit can you trust most, and which
least? Why does the integral term matter: what would the fit do without it?

**Demo: practice sequences and the c-p trade-off.** To make event times that follow the Omori-Utsu law, we use
**inverse transform sampling** (glossary `inverse_transform_sampling`): put uniform random numbers between 0
and 1 into the inverse of the **cumulative distribution function** (glossary `cumulative_distribution_function`,
CDF: the probability of a value at or below a given one) of the event times. The **parameter trade-off**
(glossary `parameter_trade_off`) shows in the contour plot: a long, tilted valley means that the data cannot
tell $c$ and $p$ apart well. Its contour lines lie 0.5, 2, 4.5, 8 and 12.5 above the lowest value of $-\ln L$,
the "1 to 5 sigma" levels: for a single parameter, a rise of $k^2/2$ above the minimum marks the edge of the
range within $k$ standard errors of the best value (glossary `standard_error`), which is what "$k$ sigma" means.

In [ ]:
def omori_times(K, c, p, t0, t1, rng):
    """Practice event times (days) of an Omori-Utsu sequence on [t0, t1], by inverse transform sampling."""
    expected = K * (np.log((t1 + c) / (t0 + c)) if p == 1 else
                    ((t1 + c) ** (1 - p) - (t0 + c) ** (1 - p)) / (1 - p))
    u = rng.random(rng.poisson(expected))
    if p == 1:
        return np.sort((t0 + c) * ((t1 + c) / (t0 + c)) ** u - c)
    lo, hi = (t0 + c) ** (1 - p), (t1 + c) ** (1 - p)
    return np.sort((lo + u * (hi - lo)) ** (1 / (1 - p)) - c)


K_TRUE, C_TRUE, P_TRUE = 300.0, 0.05, 1.1
times_all = omori_times(K_TRUE, C_TRUE, P_TRUE, 0.0, 365.0, rng)
print(f"{times_all.size} practice aftershocks; true K = {K_TRUE}, c = {C_TRUE} days, p = {P_TRUE}")
for t_s in (0.01, 0.1, 1.0):
    k_f, c_f, p_f = fit_omori(times_all[times_all >= t_s], t_s, 365.0)
    print(f"start {t_s:5.2f} days: K = {k_f:7.1f}, c = {c_f:.4f} days, p = {p_f:.3f}")

t_s = 0.1
sub = times_all[times_all >= t_s]
log_c_grid = np.linspace(np.log(0.001), np.log(2.0), 60)
p_grid = np.linspace(0.9, 1.4, 60)
surface = np.empty((p_grid.size, log_c_grid.size))
for i, p_try in enumerate(p_grid):
    for j, lc in enumerate(log_c_grid):
        expect_k1 = omori_negloglik((1.0, np.exp(lc), p_try), np.empty(0), t_s, 365.0)   # no events: the integral
        surface[i, j] = omori_negloglik((sub.size / expect_k1, np.exp(lc), p_try), sub, t_s, 365.0)
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
levels = surface.min() + np.array([0.5, 2.0, 4.5, 8.0, 12.5])          # 1 to 5 "sigma" steps
ax.contour(np.exp(log_c_grid), p_grid, surface, levels=levels, colors="tab:blue")
ax.plot(C_TRUE, P_TRUE, "*", ms=12, color="tab:red", label="True (c, p)")
ax.set(xscale="log", xlabel="c (days)", ylabel="p", title=f"Fit from {t_s} days: likelihood valley")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=1, frameon=False)
plt.show()

**Demo: an aftershock rate you can plot.** A rate is a number of events per unit of time, here events per day.
Aftershock times crowd into the first hours, so bins of equal width in time would put almost every event into
the first bin. Use bins whose edges are evenly spaced in log time instead: each bin is the same factor wider
than the one before it. Count the events in each bin and divide the count by the width of the bin in days: the
result is the average rate in that bin, which you can put next to $K/(t+c)^p$. Draw each rate at the center of
its bin on log axes; for bins evenly spaced in log time, the center is the geometric mean of the two edges,
$\sqrt{t_{\text{left}}\,t_{\text{right}}}$, the middle of the bin on a log axis. Here: the practice sequence
above, whose true rate we know.

In [ ]:
bin_edges = np.geomspace(0.001, 365.0, 26)                  # 25 bins from 0.001 to 365 days
per_bin, _ = np.histogram(times_all, bins=bin_edges)        # events in each bin
width_days = np.diff(bin_edges)
centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])           # the middle of each bin on a log axis
filled = per_bin > 0                                         # an empty bin has no point on a log axis
t_line = np.geomspace(0.001, 365.0, 200)
print(f"{per_bin.sum()} of {times_all.size} events in the bins | first bin: {per_bin[0]} events in "
      f"{width_days[0] * 1440:.1f} minutes | last bin: {per_bin[-1]} events in {width_days[-1]:.0f} days")

fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.loglog(centers[filled], per_bin[filled] / width_days[filled], "o", ms=4, color="tab:blue",
          label="Count / bin width")
ax.loglog(t_line, K_TRUE / (t_line + C_TRUE) ** P_TRUE, "-", color="tab:red", label="True rate $K/(t+c)^p$")
ax.set(xlabel="Days since the mainshock", ylabel="Events per day", title="Practice sequence: aftershock rate")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False)
plt.show()

## Part 6 · The 2016 Gyeongju aftershocks (task 6, about 120 min)

On 12 September 2016 an ML 5.8 earthquake struck near Gyeongju, the largest instrumentally recorded earthquake
in South Korea; an ML 5.1 foreshock came about 50 minutes earlier and an ML 4.5 aftershock a week later
(Hong et al. 2017; Woo et al. 2019). The named sequence `gyeongju2016` holds the catalog events within 15 km
of the epicenter from 2016-09-01 to 2017-09-30.

In task 6 you choose the completeness magnitude of this sequence yourself. First look at the counts per bin
near ML 2.0 (all events of the box). Do you see a step like the one of Part 1?

In [ ]:
gyeongju = st.data.sequence(SEQUENCE)
print(gyeongju.attrs["sequence"]["description"], "|", len(gyeongju), "events |", gyeongju.attrs["source"])
print(gyeongju["ml"].round(1).value_counts().sort_index().loc[1.4:2.6].to_string())

### M05-T6 · The 2016 Gyeongju aftershocks · core · about 120 min

**Goal:** write `aftershock_summary(seq, dm, mc, dmc, t_start, t_end)`: find the mainshock and its aftershocks in
a sequence table and report Mc, b with its uncertainty, b-positive and the Omori-Utsu parameters, with a
two-panel figure, all computed with **your** functions from tasks 2-5. Then choose the completeness magnitude
`GJ_MC` of the Gyeongju aftershocks yourself, from your own numbers.

**Given:** `st.data.sequence(SEQUENCE)`; your `mc_maxc`, `b_utsu`, `shi_bolt`, `b_positive`, `fit_omori`.

**Contract:**
- `seq`: a table like `st.data.sequence(...)` (columns `event_id`, `origin_utc` in UTC, `ml`, ...).
- The mainshock is the event with the largest `ml` (the earliest one if tied). The aftershocks are all events
  of `seq` with a later `origin_utc`, in time order.
- Return a dict with these keys:
  - `events`: the aftershock rows of `seq` (a DataFrame), in time order;
  - `mc_maxc`: your `mc_maxc` of the aftershock magnitudes with bin width `dm` and the default correction;
  - `mc`: the `mc` you were given, the completeness magnitude of everything below;
  - `b`, `sigma_b`: your `b_utsu` of the aftershock magnitudes at `mc`, and your `shi_bolt` of those at or
    above `mc`;
  - `b_positive`: your `b_positive` of the aftershock magnitudes at or above `mc`, in time order, with `dmc`;
  - `K`, `c`, `p`: your `fit_omori` of the times (in days after the mainshock) of the aftershocks at or above
    `mc` with `t_start <= t <= t_end`;
  - `t_start`, `t_end`: the window you were given (days);
  - `figure`: a matplotlib Figure with two panels. (a) The frequency-magnitude distribution of the
    aftershocks on a log scale, with `mc` marked and your Gutenberg-Richter line. (b) The rate of the
    aftershocks you fitted: their counts in bins whose edges are evenly spaced in log time, each divided by
    its bin width in days, drawn as points at the bin centers, and your fitted rate $K/(t+c)^p$ as a line,
    both axes logarithmic (the last demo of Part 5 shows such a rate). House style: sentence-case labels with
    units, legends above the panels, panel letters.
- Use the arguments, not `GJ_MC` or the values of the parameters cell: the check also runs your function on a
  made-up sequence with other values. Do not use SeismoStats inside the function.
- `GJ_MC`: your completeness magnitude for the Gyeongju aftershocks, set in the second code cell below after a
  first look at your own numbers: one value, a whole number of bins.

**Done when** `st.check("M05-T6", func=aftershock_summary, mc=GJ_MC, dmc=GJ_DMC, t_start=GJ_T_START_DAYS,
t_end=GJ_T_END_DAYS)` passes. The check runs your function on a made-up sequence (known $b$ and $p$; mc 1.5,
dmc 0.2, window 0.01-365 days), then on `gyeongju2016` with your `GJ_MC` and parameters. It compares your
numbers with SeismoStats run on the events your function returned, checks that `GJ_MC` is a completeness
magnitude that can be defended for this sequence (a whole number of bins with at least 50 aftershocks at or
above it, whose magnitudes pass a statistical test of the Gutenberg-Richter law with your own $b$; the check has
no Mc of its own), that the fitted rate adds up to the number of fitted events,
that the points of panel (b) are your counts per bin divided by the bin widths and its line is your fitted
rate, and that $b$ and $p$ lie in broad physical ranges. It never prints the real-data values.

**Stuck?** `st.hint("M05-T6", 1)`

In [ ]:
# Before you code: what Mc, b and p do you expect for the Gyeongju aftershocks, and which of your choices
# (GJ_MC, GJ_T_START_DAYS) will move them most?
st.predict("M05-T6", """

""")

In [ ]:
def aftershock_summary(seq, dm, mc, dmc, t_start, t_end):
    """Completeness, b-values and Omori-Utsu fit of the aftershocks of the largest event in seq.

    seq     : DataFrame like st.data.sequence(...)
    dm      : float, magnitude bin width
    mc      : float, completeness magnitude for b, b-positive and the Omori-Utsu fit
    dmc     : float, smallest magnitude difference for b-positive
    t_start, t_end : float, window of the Omori-Utsu fit, days after the mainshock
    returns : dict with keys events, mc_maxc, mc, b, sigma_b, b_positive, K, c, p, t_start, t_end, figure
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T6: write your code here")

**Choose your completeness magnitude.** Run your function once with a first guess for Mc (your
maximum-curvature value of all events in the box), then compare the counts per bin above with how $b$ changes
over the trial values of Mc in the table. Then set `GJ_MC` in the next cell.

In [ ]:
first_guess = mc_maxc(gyeongju["ml"].to_numpy(), DM)          # a first guess only
look = aftershock_summary(gyeongju, DM, first_guess, GJ_DMC, GJ_T_START_DAYS, GJ_T_END_DAYS)
after_ml = look["events"]["ml"].to_numpy()
print(f"{after_ml.size} aftershocks | first guess {first_guess:.1f} | your mc_maxc of the aftershocks "
      f"{look['mc_maxc']:.1f}")
for m0 in np.round(np.arange(1.3, 2.81, DM), 1):
    used = after_ml[after_ml >= m0 - DM / 2]
    b0 = b_utsu(after_ml, m0, DM)
    print(f"trial Mc {m0:.1f}: {used.size:5d} events, b = {b0:.2f} +- {shi_bolt(used, b0):.2f}")
plt.show()

In [ ]:
# Set GJ_MC, your completeness magnitude for the Gyeongju aftershocks: one value, a whole number of bins.
# YOUR CODE HERE
raise NotImplementedError("M05-T6: write your code here")

In [ ]:
st.check("M05-T6", func=aftershock_summary, mc=GJ_MC, dmc=GJ_DMC, t_start=GJ_T_START_DAYS,
         t_end=GJ_T_END_DAYS)

**Explain (3-5 sentences):** Report your Mc, $b \pm \sigma_b$, b-positive and $p$ from the cell below, and
say why you chose `GJ_MC` (from the counts and the trial table) and `GJ_T_START_DAYS`. How much does $b$ move
between the trial values of Mc near your choice, and $p$ between the start times of the cell below, and is that
more or less than $\sigma_b$? Compare your values with those of Kim and Lee (2019) and Lee et al.
(2018) for this sequence (look them up in their abstracts; note their magnitude cut-off, time window and
method), and give one reason why yours may differ.

Use it: your numbers for Gyeongju, the figure, and how they move with your choices.

In [ ]:
summary = aftershock_summary(gyeongju, DM, GJ_MC, GJ_DMC, GJ_T_START_DAYS, GJ_T_END_DAYS)
used_n = int((summary["events"]["ml"].to_numpy() >= GJ_MC - DM / 2).sum())
print(f"{len(summary['events'])} aftershocks; MAXC + 0.2 = {summary['mc_maxc']:.1f}; Mc used {summary['mc']:.1f} "
      f"({used_n} events)")
print(f"b = {summary['b']:.2f} +- {summary['sigma_b']:.2f} | b-positive (dmc {GJ_DMC}) = {summary['b_positive']:.2f}")
print(f"Omori-Utsu from {GJ_T_START_DAYS} to {GJ_T_END_DAYS} days: K = {summary['K']:.1f} per day, "
      f"c = {summary['c']:.4f} days, p = {summary['p']:.2f}")
for ext in ("png", "pdf"):
    summary["figure"].savefig(Path(FIGURE_DIR) / f"gyeongju2016_aftershocks.{ext}")
plt.show()

aftershock_mags = summary["events"]["ml"].to_numpy()
days_after = ((summary["events"]["origin_utc"] - gyeongju.loc[gyeongju["ml"].idxmax(), "origin_utc"])
              .dt.total_seconds().to_numpy() / 86400.0)
for t_s in (0.01, 0.1, 1.0):
    pick = (aftershock_mags >= GJ_MC - DM / 2) & (days_after >= t_s) & (days_after <= GJ_T_END_DAYS)
    k_f, c_f, p_f = fit_omori(days_after[pick], t_s, GJ_T_END_DAYS)
    print(f"start {t_s:4.2f} days: {pick.sum():4d} events, c = {c_f:.4f} days, p = {p_f:.2f}")
print("Source:", gyeongju.attrs["source"])

## Part 7 · Stretch: declustering the national catalog (about 90 min)

Many uses of a catalog, such as hazard estimates or a national b-value, need the **background seismicity**
(glossary `background_seismicity`): the events that were not triggered by earlier ones. **Declustering**
(glossary `declustering`) removes aftershocks (and foreshocks) to get it. The oldest recipe is that of Gardner and
Knopoff (1974): around every event, a space window and a time window that grow with its magnitude; smaller
events inside both windows of a larger one belong to its cluster and are removed. SeismoStats implements it as
`GardnerKnopoffType1` with `GardnerKnopoffWindow`.

SeismoStats needs a table with the columns `time`, `magnitude`, `latitude`, `longitude` and a fresh index
0, 1, 2, ... Its `Catalog` class silently drops time zones, so convert the times to naive UTC yourself
(`.dt.tz_convert("UTC").dt.tz_localize(None)`) and never mix UTC and KST.

In [ ]:
from seismostats.analysis import GardnerKnopoffType1, GardnerKnopoffWindow

window = GardnerKnopoffWindow()
space_km, time_windows = window(np.array([2.5, 3.5, 4.5, 5.5]))
time_days = np.array([w / pd.Timedelta(days=1) for w in time_windows])   # SeismoStats gives Timedelta objects
print(pd.DataFrame({"ML": [2.5, 3.5, 4.5, 5.5], "space window (km)": np.round(space_km, 1),
                    "time window (days)": np.round(time_days, 1)}).to_string(index=False))

### M05-T7 · Declustering the national catalog · stretch · about 90 min

**Goal:** write `decluster_national(cat, years, mc, dm)`: select the catalog events of the given UTC years at
or above `mc`, decluster them with SeismoStats' Gardner-Knopoff windows, and return $b$ before and after
declustering with a figure.

**Given:** `st.data.kma_catalog()`; `GardnerKnopoffType1`, `GardnerKnopoffWindow` (demo above); your `b_utsu`
or SeismoStats' `ClassicBValueEstimator`.

**Contract:**
- `cat`: a table like `st.data.kma_catalog()`; `years`: `(first, last)` UTC years, both included; `mc`:
  completeness magnitude (for the KMA catalog since 2018: ML 2.1 or more, never across ML 2.0); `dm`: bin width.
- Return a dict with these keys:
  - `events`: the selected rows (UTC years, `ml >= mc - dm/2`), in time order, with a fresh index 0, 1, 2, ...;
  - `mainshock_flags`: a numpy bool array, one per row of `events`, True for the events that
    `GardnerKnopoffType1(GardnerKnopoffWindow())` keeps;
  - `mc`: the `mc` you were given;
  - `b_all`, `b_declustered`: $b$ at `mc` of all selected events and of the kept ones (the Aki-Utsu or the
    exact binned formula);
  - `figure`: a matplotlib Figure with both frequency-magnitude distributions.

**Done when** `st.check("M05-T7", func=decluster_national, mc=NATIONAL_MC)` passes: a made-up catalog with
planted clusters, then the KMA catalog for 2018-2026. Your flags must match SeismoStats on your events, and your
b-values must match your events within 1 %.

**Stuck?** `st.hint("M05-T7", 1)`

In [ ]:
# Before you code: will declustering raise or lower the national b-value, and by more than its uncertainty?
st.predict("M05-T7", """

""")

In [ ]:
def decluster_national(cat, years, mc, dm):
    """Gardner-Knopoff declustering (SeismoStats) of the events of the given UTC years at or above mc.

    cat   : DataFrame like st.data.kma_catalog()
    years : (first, last) UTC years, both included
    mc    : float, completeness magnitude
    dm    : float, magnitude bin width
    returns : dict with keys events, mainshock_flags, mc, b_all, b_declustered, figure
    """
    # YOUR CODE HERE
    raise NotImplementedError("M05-T7: write your code here")

In [ ]:
st.check("M05-T7", func=decluster_national, mc=NATIONAL_MC)

**Explain (3-5 sentences):** How many events did declustering remove, and how did $b$ and the yearly rate of
events at or above `NATIONAL_MC` change (cell below)? Is the change in $b$ larger than its uncertainty? Which
clusters lost the most events, and why can large fixed windows remove independent events too?

In [ ]:
national = decluster_national(catalog, NATIONAL_YEARS, NATIONAL_MC, DM)
flags = national["mainshock_flags"]
print(f"{len(national['events'])} events of ML {NATIONAL_MC} or more; {flags.sum()} kept, {(~flags).sum()} removed")
print(f"b all {national['b_all']:.2f} +- {shi_bolt(national['events']['ml'].to_numpy(), national['b_all']):.2f} | "
      f"b declustered {national['b_declustered']:.2f}")
years_utc = national["events"]["origin_utc"].dt.year
print(pd.DataFrame({"all": years_utc.value_counts().sort_index(),
                    "kept": years_utc[flags].value_counts().sort_index()}).fillna(0).astype(int).T.to_string())
for ext in ("png", "pdf"):
    national["figure"].savefig(Path(FIGURE_DIR) / f"national_declustering.{ext}")
plt.show()

## Part 8 · Research tier (optional, not checked)

**Nearest-neighbor clustering** (glossary `nearest_neighbor_distance`; Zaliapin and Ben-Zion 2013). Instead of
fixed windows, give every event $j$ a "parent": the earlier event $i$ with the smallest proximity

$$\eta_{ij} = t_{ij}\, r_{ij}^{\,d_f}\, 10^{-b\,m_i},$$

with $t_{ij}$ the time between them (years), $r_{ij}$ the epicentral distance (km), $d_f \approx 1.6$ a fractal
dimension of the epicenters and $b \approx 1$. A histogram of $\log_{10}\eta$ often has two humps: small values
(clustered events, close to a parent) and large ones (background). The optional cell below draws it for the
national selection of task 7.

**ETAS** (glossary `etas`; Ogata 1988). The Epidemic-Type Aftershock Sequence model lets every event trigger its
own Omori-Utsu sequence, with a productivity that grows with magnitude:

$$\lambda(t) = \mu + \sum_{t_i < t} \frac{K\,10^{\alpha\,(m_i - M_c)}}{(t - t_i + c)^p}.$$

Here $\mu$ is the background rate. It is fitted with the same kind of likelihood as task 5, and it is the base of
most short-term earthquake forecasts. If you want to go further, fit it to the Gyeongju sequence and compare
the background rate with what Gardner-Knopoff keeps.

In [ ]:
# Optional: nearest-neighbor proximity for the events of task 7 (takes a few seconds).
from pyproj import Geod

ev = national["events"]
years_since = (ev["origin_utc"] - ev["origin_utc"].iloc[0]).dt.total_seconds().to_numpy() / (365.25 * 86400.0)
lat, lon, mag = (ev[k].to_numpy(dtype=float) for k in ("latitude", "longitude", "ml"))
geod = Geod(ellps="WGS84")
log_eta = np.full(len(ev), np.nan)
for j in range(1, len(ev)):
    _, _, metres = geod.inv(np.full(j, lon[j]), np.full(j, lat[j]), lon[:j], lat[:j])
    r_km = np.maximum(metres / 1000.0, 0.1)              # 0.1 km floor: catalog locations are rounded
    gap = np.maximum(years_since[j] - years_since[:j], 1e-6)
    log_eta[j] = np.min(np.log10(gap) + 1.6 * np.log10(r_km) - 1.0 * mag[:j])
fig, ax = plt.subplots(figsize=(5.2, 3.8), layout="constrained")
ax.hist(log_eta[np.isfinite(log_eta)], bins=40, color="tab:blue")
ax.set(xlabel="log10 of the nearest-neighbor proximity", ylabel="Number of events",
       title="Clustered (left) and background (right)")
plt.show()

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M05")

### Going further (optional, not checked)

1. Repeat task 6 for `pohang2017` (set `SEQUENCE` and choose its Mc as you did for Gyeongju). Is Pohang's $p$
   different from Gyeongju's?
2. Make the Monte Carlo of Part 2 for b-positive with `dmc` = 0.1, 0.2 and 0.3. How does its scatter change?
3. In the gradual-detection demo, change `DETECT_WIDTH` to 0.1 and 0.3. When does MAXC + 0.2 work well?

Your figure files stay in `figures/` inside your work folder. Commit your notebook as you learned in M00, and
log your AI conversations in `work/ai_log/M05.md`. Keep your functions: later modules use Mc and b-values of
real catalogs. Next: M03 (getting seismic data from STP), then M10 (seismograms).